# <font color="#418FDE" size="6.5" uppercase>**Backpropagation Basics**</font>

>Last update: 20260710.
    
By the end of this Lecture, you will be able to:
- Explain backpropagation as distributing output error backward through network layers. 
- Implement a simple neural network training loop with manual gradients and NumPy. 
- Analyze neural network learning using loss curves, test metrics, and overfitting checks. 


## **1. Gradient Flow**

### **1.1. Output Error**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_07/Lecture_B/image_01_01.jpg?v=1783671887" width="250">



>* Compare predictions with targets to measure loss
>* Output error guides needed prediction changes

>* Output error turns performance into training feedback
>* Loss signals guide backward responsibility tracing

>* Output error starts backward responsibility flow
>* Influential parameters receive stronger adjustment signals



In [ ]:
#@title Python Code - Output Error

# Output error starts gradient flow.
# Tiny network predicts beam deflection.
# Manual gradients show responsibility sharing.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable learning.
rng = np.random.default_rng(7)

# Create tiny mechanical engineering training data.
load_kN = np.array([[1.0], [2.0], [3.0], [4.0]])
true_mm = np.array([[2.0], [4.1], [5.9], [8.2]])

# Scale values to keep gradients stable.
x = load_kN / 4.0
y = true_mm / 8.2

# Validate matching sample and target shapes.
assert x.shape == y.shape
assert x.shape[0] == 4

# Initialize one hidden layer network parameters.
W1 = rng.normal(0.0, 0.4, size=(1, 3))
b1 = np.zeros((1, 3))
W2 = rng.normal(0.0, 0.4, size=(3, 1))
b2 = np.zeros((1, 1))

# Store losses for one compact curve.
losses = []
learning_rate = 0.35

# Train silently with manual backpropagation.
for epoch in range(120):
    hidden_raw = x @ W1 + b1
    hidden = np.tanh(hidden_raw)
    prediction = hidden @ W2 + b2

    output_error = prediction - y
    loss = np.mean(output_error ** 2)
    losses.append(loss)

    d_prediction = 2.0 * output_error / y.size
    dW2 = hidden.T @ d_prediction
    db2 = np.sum(d_prediction, axis=0, keepdims=True)

    d_hidden = d_prediction @ W2.T
    d_hidden_raw = d_hidden * (1.0 - hidden ** 2)
    dW1 = x.T @ d_hidden_raw
    db1 = np.sum(d_hidden_raw, axis=0, keepdims=True)

    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

# Recompute final predictions and output errors.
hidden_raw = x @ W1 + b1
hidden = np.tanh(hidden_raw)
prediction = hidden @ W2 + b2
output_error = prediction - y

# Convert scaled predictions back to millimeters.
pred_mm = prediction * 8.2
err_mm = output_error * 8.2

# Print only compact teaching results.
print(f"NumPy version: {np.__version__}")
print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Last output error: {err_mm[-1, 0]:+.3f} mm")
print("Output error is the first backward signal.")

# Plot loss curve to show learning progress.
plt.figure(figsize=(6, 3.5))
plt.plot(losses, color="navy", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("Output Error Shrinks During Backpropagation")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### **1.2. Hidden Layer Gradients**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_07/Lecture_B/image_01_02.jpg?v=1783671890" width="250">



>* Hidden gradients assign responsibility to earlier layers
>* Backpropagation traces errors backward for better updates

>* Hidden neurons pass pattern signals forward
>* Backward gradients measure loss sensitivity

>* Hidden gradients build useful internal features
>* Deep networks can weaken or distort gradients



In [ ]:
#@title Python Code - Hidden Layer Gradients

# Hidden gradients distribute output error backward.
# This tiny network uses manual NumPy gradients.
# Printed values show responsibility reaching hidden units.

# NumPy and Matplotlib are already available.
import numpy as np
import matplotlib.pyplot as plt

# Use deterministic randomness for repeatable teaching results.
rng = np.random.default_rng(7)

# Create tiny mechanical-style sensor data.
X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = np.array([[0.0], [1.0], [1.0], [0.0]])

# Validate the small training shapes.
assert X.shape == (4, 2)
assert y.shape == (4, 1)

# Initialize a two-layer neural network.
W1 = rng.normal(0.0, 0.8, size=(2, 3))
b1 = np.zeros((1, 3))
W2 = rng.normal(0.0, 0.8, size=(3, 1))
b2 = np.zeros((1, 1))

# Define sigmoid activation and derivative.
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_grad(a):
    return a * (1.0 - a)

# Store loss values for one compact plot.
losses = []
learning_rate = 0.8

# Train silently with manual backpropagation.
for epoch in range(1200):
    z1 = X @ W1 + b1
    h = sigmoid(z1)
    z2 = h @ W2 + b2
    yhat = sigmoid(z2)

    error = yhat - y
    loss = np.mean(error ** 2)
    losses.append(loss)

    d_yhat = 2.0 * error / y.size
    d_z2 = d_yhat * sigmoid_grad(yhat)
    d_W2 = h.T @ d_z2
    d_b2 = np.sum(d_z2, axis=0, keepdims=True)

    d_h = d_z2 @ W2.T
    d_z1 = d_h * sigmoid_grad(h)
    d_W1 = X.T @ d_z1
    d_b1 = np.sum(d_z1, axis=0, keepdims=True)

    W2 -= learning_rate * d_W2
    b2 -= learning_rate * d_b2
    W1 -= learning_rate * d_W1
    b1 -= learning_rate * d_b1

# Compute final predictions and gradient magnitudes.
final_predictions = (yhat >= 0.5).astype(int)
hidden_signal = np.mean(np.abs(d_z1), axis=0)
output_signal = float(np.mean(np.abs(d_z2)))

# Print only compact teaching results.
print(f"NumPy version: {np.__version__}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Predicted classes: {final_predictions.ravel().tolist()}")
print(f"Mean output gradient signal: {output_signal:.6f}")
print(f"Hidden gradient signals: {np.round(hidden_signal, 6).tolist()}")
print("Hidden gradients come from output error through W2.")

# Plot the loss curve to show learning.
plt.figure(figsize=(6, 3.5))
plt.plot(losses, color="navy", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("Manual Backpropagation Loss Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### **1.3. Chain Rule**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_07/Lecture_B/image_01_03.jpg?v=1783671892" width="250">



>* Chain rule links earlier weights to error
>* Backprop combines local effects backward

>* Trace errors backward through earlier stages
>* Assign larger gradients to stronger contributors

>* Chain rule updates early deep-network layers
>* Gradient size affects learning stability



In [ ]:
#@title Python Code - Chain Rule

# Chain rule explains backward responsibility.
# Tiny network shows gradient flow.
# NumPy computes every local derivative.

# Import NumPy and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Set deterministic values for reproducibility.
np.random.seed(7)

# Create one mechanical sensor example.
x = np.array([[2.0]])
y_true = np.array([[1.0]])

# Define two weights in series.
w1 = np.array([[0.40]])
w2 = np.array([[1.50]])

# Run the forward pass.
hidden = x @ w1
prediction = hidden @ w2

# Compute squared error loss.
error = prediction - y_true
loss = 0.5 * error**2

# Backpropagate using local derivatives.
dloss_dpred = error
dpred_dw2 = hidden
dpred_dhidden = w2

# Continue the chain backward.
dhidden_dw1 = x
dloss_dw2 = dpred_dw2.T @ dloss_dpred
dloss_dw1 = dhidden_dw1.T @ (dloss_dpred @ dpred_dhidden.T)

# Check expected gradient shapes.
assert dloss_dw1.shape == w1.shape
assert dloss_dw2.shape == w2.shape

# Show compact numerical results.
print(f"Prediction: {prediction.item():.3f}")
print(f"Loss: {loss.item():.3f}")
print(f"Gradient for w2: {dloss_dw2.item():.3f}")
print(f"Gradient for w1: {dloss_dw1.item():.3f}")
print("w1 gradient = output error × w2 × input")

# Visualize the backward responsibility chain.
labels = ["output error", "through w2", "to w1"]
values = [dloss_dpred.item(), (dloss_dpred @ dpred_dhidden.T).item(), dloss_dw1.item()]

# Draw one simple gradient flow plot.
plt.figure(figsize=(6, 3))
plt.bar(labels, values, color=["tomato", "orange", "steelblue"])
plt.ylabel("Gradient signal")
plt.title("Chain Rule: Error Signal Moving Backward")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()



## **2. Training Loop**

### **2.1. Forward Pass**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_07/Lecture_B/image_02_01.jpg?v=1783671881" width="250">



>* Inputs flow through layers to predictions
>* Current weights shape the model’s guess

>* Save forward-pass values for later gradients
>* Stored values show how parameters caused loss

>* Loss compares predictions with true targets
>* Repeated passes reveal learning progress



In [ ]:
#@title Python Code - Forward Pass

# Demonstrate one neural network forward pass.
# Store intermediate values for later gradients.
# Use tiny mechanical data with NumPy.

import numpy as np

# Set deterministic values for repeatable teaching.
rng = np.random.default_rng(seed=7)

# Create two beam examples with simple features.
X = np.array([[10.0, 2.0], [20.0, 3.0]])

# Scale features for stable matrix calculations.
X_scaled = X / np.array([[20.0, 3.0]])

# Define small weights and biases manually.
W1 = rng.normal(0.0, 0.3, size=(2, 3))
b1 = np.zeros((1, 3))

# Define output layer parameters manually.
W2 = rng.normal(0.0, 0.3, size=(3, 1))
b2 = np.zeros((1, 1))

# Check shapes before matrix multiplication.
assert X_scaled.shape[1] == W1.shape[0]
assert W1.shape[1] == W2.shape[0]

# Compute hidden pre-activation values.
Z1 = X_scaled @ W1 + b1

# Apply ReLU activation to hidden values.
A1 = np.maximum(0.0, Z1)

# Compute final prediction values.
y_pred = A1 @ W2 + b2

# Store forward-pass values for backpropagation.
cache = {"X": X_scaled, "Z1": Z1, "A1": A1, "y_pred": y_pred}

# Compare predictions with tiny target deflections.
y_true = np.array([[0.12], [0.28]])
loss = np.mean((y_pred - y_true) ** 2)

# Print compact teaching results only.
print("NumPy version:", np.__version__)
print("Input shape:", X_scaled.shape)
print("Hidden activation shape:", A1.shape)
print("Prediction shape:", y_pred.shape)
print("Predictions:", np.round(y_pred.ravel(), 4))
print("Mean squared error:", round(float(loss), 6))
print("Cached values:", sorted(cache.keys()))



### **2.2. Backward Gradients**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_07/Lecture_B/image_02_02.jpg?v=1783671883" width="250">



>* Trace errors backward through network layers
>* Gradients show each parameter’s influence

>* Trace gradients backward through connected layers
>* Activation behavior controls error signal flow

>* Match gradient shapes and stored values
>* Average batch gradients to guide learning



In [ ]:
#@title Python Code - Backward Gradients

# Manual gradients reveal backward error flow.
# Tiny NumPy network keeps learning visible.
# Loss curves help check overfitting.

# NumPy and Matplotlib are already available.
import numpy as np
import matplotlib.pyplot as plt

# Use deterministic randomness for repeatable teaching results.
rng = np.random.default_rng(7)

# Create tiny mechanical sensor data.
n_samples = 80
x_force = rng.normal(0.0, 1.0, n_samples)

# Add a second feature with different scale.
x_temp = rng.normal(0.0, 1.0, n_samples)
X = np.column_stack((x_force, x_temp))

# Define a nonlinear pass or fail target.
y_raw = x_force + 0.8 * x_temp + 0.5 * x_force * x_temp
y = (y_raw > 0.2).astype(float).reshape(-1, 1)

# Split data into training and testing sets.
X_train, X_test = X[:60], X[60:]
y_train, y_test = y[:60], y[60:]

# Validate shapes before matrix operations.
assert X_train.shape == (60, 2)
assert y_train.shape == (60, 1)

# Initialize a small two layer network.
W1 = rng.normal(0.0, 0.4, (2, 4))
b1 = np.zeros((1, 4))

# Initialize output layer parameters.
W2 = rng.normal(0.0, 0.4, (4, 1))
b2 = np.zeros((1, 1))

# Define stable sigmoid activation function.
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Store losses for one compact curve.
train_losses = []
test_losses = []

# Train silently with manual backward gradients.
learning_rate = 0.25
epochs = 300

# Run a short full batch training loop.
for epoch in range(epochs):
    z1 = X_train @ W1 + b1
    a1 = np.tanh(z1)

    z2 = a1 @ W2 + b2
    y_hat = sigmoid(z2)

    eps = 1e-9
    train_loss = -np.mean(y_train * np.log(y_hat + eps) + (1 - y_train) * np.log(1 - y_hat + eps))

    # Start backward pass at output error.
    dz2 = (y_hat - y_train) / len(y_train)
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    # Move error backward through hidden layer.
    da1 = dz2 @ W2.T
    dz1 = da1 * (1 - a1 ** 2)
    dW1 = X_train.T @ dz1

    # Bias gradients summarize hidden shifts.
    db1 = np.sum(dz1, axis=0, keepdims=True)
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    # Update earlier layer after its gradients.
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    # Track train and test loss occasionally.
    if epoch % 10 == 0:
        train_losses.append(train_loss)
        test_a1 = np.tanh(X_test @ W1 + b1)

        test_hat = sigmoid(test_a1 @ W2 + b2)
        test_loss = -np.mean(y_test * np.log(test_hat + eps) + (1 - y_test) * np.log(1 - test_hat + eps))
        test_losses.append(test_loss)

# Compute final predictions and accuracy.
train_pred = (y_hat >= 0.5).astype(float)
test_a1 = np.tanh(X_test @ W1 + b1)

# Evaluate the held out examples.
test_hat = sigmoid(test_a1 @ W2 + b2)
test_pred = (test_hat >= 0.5).astype(float)

# Summarize learning without printing arrays.
train_acc = np.mean(train_pred == y_train)
test_acc = np.mean(test_pred == y_test)

# Print compact teaching results.
print(f"NumPy version: {np.__version__}")
print(f"Final train loss: {train_losses[-1]:.3f}")
print(f"Final test loss: {test_losses[-1]:.3f}")
print(f"Train accuracy: {train_acc:.2f}")
print(f"Test accuracy: {test_acc:.2f}")
print(f"Gradient shapes: dW1 {dW1.shape}, dW2 {dW2.shape}")

# Plot loss curves for overfitting checks.
plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="train loss")
plt.plot(test_losses, label="test loss")
plt.xlabel("Checkpoint every 10 epochs")
plt.ylabel("Cross entropy loss")
plt.title("Manual Backward Gradients Training Loop")
plt.legend()
plt.tight_layout()
plt.show()



### **2.3. Weight Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_07/Lecture_B/image_02_03.jpg?v=1783671885" width="250">



>* Updates adjust parameters using gradient feedback
>* Manual NumPy changes help reduce future loss

>* Learning rate controls update size
>* Loss trends reveal update stability

>* Updates gradually reshape learned input patterns
>* Monitor loss and overfitting for generalization



In [ ]:
#@title Python Code - Weight Updates

# Manual updates change neural network parameters.
# Gradients point toward lower prediction loss.
# Learning rate controls update step size.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable results.
rng = np.random.default_rng(7)

# Create tiny engineering-style training data.
x_raw = np.linspace(0.0, 10.0, 40).reshape(-1, 1)
y_raw = 0.8 * x_raw + 0.6 + rng.normal(0.0, 0.35, x_raw.shape)

# Normalize inputs for stable manual gradients.
x_mean, x_std = x_raw.mean(), x_raw.std()
y_mean, y_std = y_raw.mean(), y_raw.std()

# Convert data to normalized training arrays.
X = (x_raw - x_mean) / x_std
Y = (y_raw - y_mean) / y_std

# Validate shapes before matrix operations.
assert X.shape == Y.shape == (40, 1)

# Initialize one hidden layer network parameters.
W1 = rng.normal(0.0, 0.4, (1, 4))
b1 = np.zeros((1, 4))

# Initialize output layer parameters.
W2 = rng.normal(0.0, 0.4, (4, 1))
b2 = np.zeros((1, 1))

# Choose a measured learning rate.
learning_rate = 0.08
epochs = 120
losses = []

# Train using forward pass, gradients, and updates.
for epoch in range(epochs):
    hidden_linear = X @ W1 + b1
    hidden = np.tanh(hidden_linear)

    predictions = hidden @ W2 + b2
    error = predictions - Y
    loss = float(np.mean(error ** 2))

    losses.append(loss)
    d_predictions = 2.0 * error / X.shape[0]
    dW2 = hidden.T @ d_predictions

    db2 = d_predictions.sum(axis=0, keepdims=True)
    d_hidden = d_predictions @ W2.T
    d_hidden_linear = d_hidden * (1.0 - hidden ** 2)

    dW1 = X.T @ d_hidden_linear
    db1 = d_hidden_linear.sum(axis=0, keepdims=True)
    W2 -= learning_rate * dW2

    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

# Compute final predictions in original units.
final_hidden = np.tanh(X @ W1 + b1)
final_scaled = final_hidden @ W2 + b2
final_predictions = final_scaled * y_std + y_mean

# Summarize learning without printing large arrays.
rmse = float(np.sqrt(np.mean((final_predictions - y_raw) ** 2)))
print(f"NumPy version: {np.__version__}")
print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Training RMSE: {rmse:.3f} kN")

# Plot the loss curve from repeated updates.
plt.figure(figsize=(6, 4))
plt.plot(losses, color="navy", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("Loss decreases through manual weight updates")
plt.grid(True, alpha=0.3)
plt.show()



## **3. Network Diagnosis**

### **3.1. Loss Curves**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_07/Lecture_B/image_03_01.jpg?v=1783671875" width="250">



>* Loss curves track learning progress over time
>* Falling loss suggests improving model predictions

>* Compare training and validation loss
>* Widening gaps can reveal overfitting

>* Curve shapes reveal training setup problems
>* Use curves to judge deployment readiness



In [ ]:
#@title Python Code - Loss Curves

# Loss curves reveal neural network learning.
# Validation loss helps detect overfitting.
# This example uses manual NumPy gradients.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for repeatable results.
rng = np.random.default_rng(7)

# Create a small mechanical sensor dataset.
n_samples, n_features = 180, 2
X = rng.normal(size=(n_samples, n_features))

# Define labels from two noisy measurements.
noise = 0.35 * rng.normal(size=n_samples)
y = ((1.4 * X[:, 0] - X[:, 1] + noise) > 0).astype(float)

# Split data into training and validation sets.
train_size = 120
X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = y[:train_size], y[train_size:]

# Validate the expected matrix dimensions.
assert X_train.shape == (120, 2)
assert X_val.shape[1] == X_train.shape[1]

# Standardize features using training statistics only.
mean = X_train.mean(axis=0, keepdims=True)
std = X_train.std(axis=0, keepdims=True) + 1e-8
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std

# Initialize a tiny neural network manually.
hidden = 6
W1 = 0.3 * rng.normal(size=(n_features, hidden))
b1 = np.zeros((1, hidden))
W2 = 0.3 * rng.normal(size=(hidden, 1))
b2 = np.zeros((1, 1))

# Define stable sigmoid and loss functions.
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def binary_loss(y_true, y_prob):
    eps = 1e-8
    return -np.mean(y_true * np.log(y_prob + eps) + (1 - y_true) * np.log(1 - y_prob + eps))

# Store losses for diagnostic curves.
epochs = 160
learning_rate = 0.08
train_losses, val_losses = [], []

# Train silently using manual backpropagation.
for epoch in range(epochs):
    z1 = X_train @ W1 + b1
    a1 = np.tanh(z1)
    y_hat = sigmoid(a1 @ W2 + b2).ravel()

    error = (y_hat - y_train).reshape(-1, 1)
    dW2 = a1.T @ error / train_size
    db2 = error.mean(axis=0, keepdims=True)
    dz1 = (error @ W2.T) * (1 - a1**2)

    dW1 = X_train.T @ dz1 / train_size
    db1 = dz1.mean(axis=0, keepdims=True)
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    train_losses.append(binary_loss(y_train, y_hat))
    val_prob = sigmoid(np.tanh(X_val @ W1 + b1) @ W2 + b2).ravel()

    val_losses.append(binary_loss(y_val, val_prob))

# Compute final validation accuracy.
val_pred = (val_prob >= 0.5).astype(float)
val_accuracy = np.mean(val_pred == y_val)

# Print a compact diagnosis summary.
print(f"NumPy version: {np.__version__}")
print(f"Final training loss: {train_losses[-1]:.3f}")
print(f"Final validation loss: {val_losses[-1]:.3f}")
print(f"Validation accuracy: {val_accuracy:.2%}")
print("Large train-validation gaps can warn about overfitting.")

# Plot training and validation loss curves.
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="training loss")
plt.plot(val_losses, label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Loss Curves for Network Diagnosis")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **3.2. Loss Curve Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_07/Lecture_B/image_03_02.jpg?v=1783671879" width="250">



>* Track loss to see learning progress
>* Compare validation loss to confirm generalization

>* Widening loss gaps often signal overfitting
>* High losses may indicate underfitting problems

>* Spot unstable loss and optimization problems
>* Use curves to guide training decisions



In [ ]:
#@title Python Code - Loss Curve Checks

# Diagnose learning with loss curves.
# Compare training and validation behavior.
# Use NumPy for manual gradients.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable results.
rng = np.random.default_rng(7)

# Create a small mechanical sensor dataset.
n_samples = 160
x = rng.uniform(-3.0, 3.0, size=(n_samples, 1))

# Model a nonlinear vibration response.
noise = rng.normal(0.0, 0.12, size=(n_samples, 1))
y = np.sin(1.7 * x) + 0.25 * x + noise

# Split data into training and validation sets.
order = rng.permutation(n_samples)
train_idx = order[:110]
val_idx = order[110:]

# Standardize inputs using training statistics.
x_mean = x[train_idx].mean(axis=0, keepdims=True)
x_std = x[train_idx].std(axis=0, keepdims=True) + 1e-8
x_scaled = (x - x_mean) / x_std

# Prepare train and validation arrays.
X_train = x_scaled[train_idx]
y_train = y[train_idx]
X_val = x_scaled[val_idx]
y_val = y[val_idx]

# Validate shapes before training begins.
assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]

# Initialize a tiny one hidden layer network.
hidden = 12
W1 = rng.normal(0.0, 0.4, size=(1, hidden))
b1 = np.zeros((1, hidden))

# Initialize output layer parameters.
W2 = rng.normal(0.0, 0.4, size=(hidden, 1))
b2 = np.zeros((1, 1))

# Store losses for curve diagnosis.
epochs = 220
lr = 0.035
train_losses = []
val_losses = []

# Train silently with manual backpropagation.
for epoch in range(epochs):
    z1 = X_train @ W1 + b1
    a1 = np.tanh(z1)
    pred = a1 @ W2 + b2

    error = pred - y_train
    train_loss = float(np.mean(error ** 2))
    train_losses.append(train_loss)

    grad_pred = 2.0 * error / len(X_train)
    grad_W2 = a1.T @ grad_pred
    grad_b2 = grad_pred.sum(axis=0, keepdims=True)

    grad_a1 = grad_pred @ W2.T
    grad_z1 = grad_a1 * (1.0 - a1 ** 2)
    grad_W1 = X_train.T @ grad_z1

    grad_b1 = grad_z1.sum(axis=0, keepdims=True)
    W2 -= lr * grad_W2
    b2 -= lr * grad_b2

    W1 -= lr * grad_W1
    b1 -= lr * grad_b1
    val_hidden = np.tanh(X_val @ W1 + b1)
    val_pred = val_hidden @ W2 + b2

    val_error = val_pred - y_val
    val_loss = float(np.mean(val_error ** 2))
    val_losses.append(val_loss)

# Summarize final learning behavior briefly.
best_epoch = int(np.argmin(val_losses)) + 1
final_gap = val_losses[-1] - train_losses[-1]
print(f"NumPy version: {np.__version__}")

print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final validation loss: {val_losses[-1]:.4f}")
print(f"Best validation epoch: {best_epoch}")

# Give a simple diagnostic interpretation.
if final_gap > 0.08:
    diagnosis = "possible overfitting gap"
elif val_losses[-1] > val_losses[0] * 0.85:
    diagnosis = "weak learning or underfitting"
else:
    diagnosis = "reasonable generalization pattern"

print(f"Loss curve check: {diagnosis}")

# Plot both curves for visual diagnosis.
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="training loss")
plt.plot(val_losses, label="validation loss")

plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best validation")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("Loss Curve Checks for Network Diagnosis")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### **3.3. Test Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_07/Lecture_B/image_03_03.jpg?v=1783671877" width="250">



>* Measure performance on unseen data
>* Choose metrics suited to task risks

>* Keep test data for final evaluation
>* Use metrics to judge real-world generalization

>* Compare metrics to spot overfitting or underfitting
>* Check detailed errors before trusting models



In [ ]:
#@title Python Code - Test Metrics

# Test metrics judge unseen network performance.
# Manual NumPy training keeps gradients visible.
# Loss gaps help diagnose overfitting.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable results.
rng = np.random.default_rng(7)

# Create a small mechanical classification dataset.
n_samples, n_features = 240, 2
X = rng.normal(0.0, 1.0, (n_samples, n_features))

# Label unsafe parts using a nonlinear boundary.
noise = rng.normal(0.0, 0.25, n_samples)
y = ((X[:, 0] ** 2 + 0.7 * X[:, 1] + noise) > 0.9).astype(int)

# Split data into train, validation, and test.
order = rng.permutation(n_samples)
train_idx, val_idx, test_idx = order[:140], order[140:190], order[190:]

# Standardize using training statistics only.
mean, std = X[train_idx].mean(axis=0), X[train_idx].std(axis=0)
X_scaled = (X - mean) / (std + 1e-8)

# Prepare split arrays for evaluation.
X_train, y_train = X_scaled[train_idx], y[train_idx]
X_val, y_val = X_scaled[val_idx], y[val_idx]
X_test, y_test = X_scaled[test_idx], y[test_idx]

# Validate expected matrix and label shapes.
assert X_train.shape[1] == n_features and y_test.size == X_test.shape[0]

# Initialize a tiny neural network manually.
hidden = 8
W1 = rng.normal(0.0, 0.5, (n_features, hidden))
b1 = np.zeros(hidden)

# Initialize output layer parameters.
W2 = rng.normal(0.0, 0.5, (hidden, 1))
b2 = np.zeros(1)

# Define stable sigmoid and binary loss.
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

# Compute forward pass probabilities.
def forward(X_batch):
    h = np.tanh(X_batch @ W1 + b1)
    p = sigmoid(h @ W2 + b2).ravel()
    return h, p

# Compute average binary cross entropy.
def loss_value(y_true, p_pred):
    p_safe = np.clip(p_pred, 1e-8, 1 - 1e-8)
    return -np.mean(y_true * np.log(p_safe) + (1 - y_true) * np.log(1 - p_safe))

# Train silently while storing diagnostic losses.
lr, epochs = 0.08, 220
train_losses, val_losses = [], []

# Use globals for simple teaching updates.
for epoch in range(epochs):
    h, p = forward(X_train)
    error = (p - y_train).reshape(-1, 1)

    # Backpropagate output error through layers.
    dW2 = h.T @ error / X_train.shape[0]
    db2 = error.mean(axis=0)

    # Continue gradients into hidden layer.
    dh = (error @ W2.T) * (1 - h ** 2)
    dW1 = X_train.T @ dh / X_train.shape[0]
    db1 = dh.mean(axis=0)

    # Update parameters using gradient descent.
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    # Store train and validation losses.
    train_losses.append(loss_value(y_train, forward(X_train)[1]))
    val_losses.append(loss_value(y_val, forward(X_val)[1]))

# Compute classification metrics without libraries.
def metrics(y_true, p_pred):
    y_hat = (p_pred >= 0.5).astype(int)
    tp = np.sum((y_hat == 1) & (y_true == 1))
    fp = np.sum((y_hat == 1) & (y_true == 0))

    # Count missed positives and correct negatives.
    fn = np.sum((y_hat == 0) & (y_true == 1))
    tn = np.sum((y_hat == 0) & (y_true == 0))

    # Return accuracy, precision, and recall.
    acc = (tp + tn) / y_true.size
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    return acc, prec, rec

# Evaluate final model on untouched test data.
test_prob = forward(X_test)[1]
test_acc, test_prec, test_rec = metrics(y_test, test_prob)

# Compare train, validation, and test behavior.
train_acc, _, _ = metrics(y_train, forward(X_train)[1])
val_acc, _, _ = metrics(y_val, forward(X_val)[1])

# Print concise diagnostic results only.
print(f"NumPy version: {np.__version__}")
print(f"Final train loss: {train_losses[-1]:.3f}")
print(f"Final validation loss: {val_losses[-1]:.3f}")
print(f"Train accuracy: {train_acc:.2f}, validation accuracy: {val_acc:.2f}")
print(f"Test accuracy: {test_acc:.2f}, precision: {test_prec:.2f}, recall: {test_rec:.2f}")

# Plot loss curves for overfitting checks.
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="training loss")
plt.plot(val_losses, label="validation loss")

# Label the diagnostic learning curve.
plt.xlabel("Epoch")
plt.ylabel("Binary cross entropy")
plt.title("Loss curves before final test metrics")
plt.legend()

# Display exactly one compact plot.
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Backpropagation Basics**</font>


In this lecture, you learned to:
- Explain backpropagation as distributing output error backward through network layers. 
- Implement a simple neural network training loop with manual gradients and NumPy. 
- Analyze neural network learning using loss curves, test metrics, and overfitting checks. 

In the next Module (Module 8), we will go over 'None'